# Demystifying the Attention Layer in LLMs
### PyAtl Presentation - February 2024

This notebook demonstrates:
1. Tokenization and vectorization
2. Word embeddings and vector arithmetic
3. Attention mechanisms
4. How context modifies meaning

---
## Setup and Installations

In [ ]:
# Install required packages (uncomment if needed)
# !pip install transformers torch numpy matplotlib seaborn gensim bertviz scipy

In [1]:
import numpy as np
import torch
from transformers import BertTokenizer, BertModel, GPT2Tokenizer, GPT2LMHeadModel
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial.distance import cosine
import warnings
warnings.filterwarnings('ignore')

print("All libraries loaded successfully!")

/Users/trenton/github/tchoffman/demystify-attention/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All libraries loaded successfully!


---
## Part 1: Tokenization - Breaking Text into Pieces

In [2]:
# Load a tokenizer
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')

def demonstrate_tokenization(text):
    """Show how text gets broken into tokens"""
    tokens = tokenizer.tokenize(text)
    token_ids = tokenizer.encode(text)
    
    print(f"\nOriginal text: '{text}'")
    print(f"Tokens: {tokens}")
    print(f"Token IDs: {token_ids}")
    print(f"Number of tokens: {len(tokens)}")
    return tokens, token_ids

# NOTE: GPT-2 uses byte-level BPE, which maps all bytes to visible characters.
# The 'Ġ' character (Unicode \u0120) represents a leading space before a token.
# So 'Ġquick' really means ' quick'. The first token in a sequence won't have
# one since there's no preceding space.

In [3]:
# Pre-loaded examples
example_texts = [
    "Hello, world!",
    "The quick brown fox jumps over the lazy dog.",
    "Tokenization is fascinating!",
    "PyAtl is awesome!"
]

for text in example_texts:
    demonstrate_tokenization(text)


Original text: 'Hello, world!'
Tokens: ['Hello', ',', 'Ġworld', '!']
Token IDs: [15496, 11, 995, 0]
Number of tokens: 4

Original text: 'The quick brown fox jumps over the lazy dog.'
Tokens: ['The', 'Ġquick', 'Ġbrown', 'Ġfox', 'Ġjumps', 'Ġover', 'Ġthe', 'Ġlazy', 'Ġdog', '.']
Token IDs: [464, 2068, 7586, 21831, 18045, 625, 262, 16931, 3290, 13]
Number of tokens: 10

Original text: 'Tokenization is fascinating!'
Tokens: ['Token', 'ization', 'Ġis', 'Ġfascinating', '!']
Token IDs: [30642, 1634, 318, 13899, 0]
Number of tokens: 5

Original text: 'PyAtl is awesome!'
Tokens: ['Py', 'Atl', 'Ġis', 'Ġawesome', '!']
Token IDs: [20519, 25255, 318, 7427, 0]
Number of tokens: 5


In [4]:
# The leading space matters! "The" at the start of a sentence and " The" after
# a space are different tokens with different IDs in the vocabulary.
word = "The"
no_space = tokenizer.encode(word)
with_space = tokenizer.encode(" " + word)

print(f"'{word}'  -> token IDs: {no_space}  tokens: {tokenizer.tokenize(word)}")
print(f"' {word}' -> token IDs: {with_space}  tokens: {tokenizer.tokenize(' ' + word)}")
print(f"\nSame word, different tokens — position context is baked into the vocabulary.")

'The'  -> token IDs: [464]  tokens: ['The']
' The' -> token IDs: [383]  tokens: ['ĠThe']

Same word, different tokens — position context is baked into the vocabulary.


In [5]:
# Try your own!
demonstrate_tokenization('your text here')


Original text: 'your text here'
Tokens: ['your', 'Ġtext', 'Ġhere']
Token IDs: [14108, 2420, 994]
Number of tokens: 3


(['your', 'Ġtext', 'Ġhere'], [14108, 2420, 994])

In [6]:
# Compare with BERT's tokenizer (WordPiece instead of byte-level BPE)
# BERT uses '##' to mark subword continuations (instead of GPT-2's 'Ġ' for spaces)
# and bert-base-uncased lowercases everything — no case duplication.
bert_tok = BertTokenizer.from_pretrained('bert-base-uncased')

print("Same examples with BERT (WordPiece, uncased):\n")
for text in example_texts:
    tokens = bert_tok.tokenize(text)
    token_ids = bert_tok.encode(text, add_special_tokens=False)
    print(f"Original text: '{text}'")
    print(f"Tokens: {tokens}")
    print(f"Token IDs: {token_ids}")
    print(f"Number of tokens: {len(tokens)}\n")

# Notice:
# - No 'Ġ' characters — spaces are implicit between tokens
# - Everything is lowercased ("The" -> "the")
# - Unknown subwords get '##' prefix (e.g. "PyAtl" -> ["py", "##at", "##l"])
# - BERT also wraps input with special [CLS] and [SEP] tokens (omitted here)

Same examples with BERT (WordPiece, uncased):

Original text: 'Hello, world!'
Tokens: ['hello', ',', 'world', '!']
Token IDs: [7592, 1010, 2088, 999]
Number of tokens: 4

Original text: 'The quick brown fox jumps over the lazy dog.'
Tokens: ['the', 'quick', 'brown', 'fox', 'jumps', 'over', 'the', 'lazy', 'dog', '.']
Token IDs: [1996, 4248, 2829, 4419, 14523, 2058, 1996, 13971, 3899, 1012]
Number of tokens: 10

Original text: 'Tokenization is fascinating!'
Tokens: ['token', '##ization', 'is', 'fascinating', '!']
Token IDs: [19204, 3989, 2003, 17160, 999]
Number of tokens: 5

Original text: 'PyAtl is awesome!'
Tokens: ['p', '##yat', '##l', 'is', 'awesome', '!']
Token IDs: [1052, 26139, 2140, 2003, 12476, 999]
Number of tokens: 6



---
## Part 2: Word Embeddings - Words as Vectors

In [7]:
# Load pre-trained word embeddings (using BERT)
bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_model = BertModel.from_pretrained('bert-base-uncased')
bert_model.eval()

print("BERT model loaded!")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2147.29it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BERT model loaded!


In [ ]:
def get_word_embedding(word):
    """Get the static embedding vector for a single word from BERT's embedding table"""
    token_ids = bert_tokenizer.encode(word, add_special_tokens=False)
    # Use the first token's embedding (handles multi-token words gracefully)
    embedding = bert_model.embeddings.word_embeddings.weight[token_ids[0]]
    return embedding.detach().numpy()

# Pre-compute normalized embedding matrix and token list for fast full-vocab search
_embedding_matrix = bert_model.embeddings.word_embeddings.weight.detach().numpy()
_embedding_norms = np.linalg.norm(_embedding_matrix, axis=1, keepdims=True)
_normalized_embeddings = _embedding_matrix / _embedding_norms

# Build a set of token IDs to skip: subword tokens (##...), special tokens,
# numbers, punctuation, and non-Latin characters
_special_tokens = {'[CLS]', '[SEP]', '[PAD]', '[UNK]', '[MASK]'}
_skip_ids = set()
for _id in range(len(bert_tokenizer)):
    _tok = bert_tokenizer.convert_ids_to_tokens(_id)
    if _tok.startswith('##') or _tok in _special_tokens or not _tok.isascii() or not _tok.isalpha():
        _skip_ids.add(_id)

# Pre-compute the vocabulary-wide mean embedding for centering
_valid_ids = [i for i in range(len(bert_tokenizer)) if i not in _skip_ids]
_vocab_mean = _embedding_matrix[_valid_ids].mean(axis=0)

_whole_word_count = len(bert_tokenizer) - len(_skip_ids)
print(f"Vocab size: {len(bert_tokenizer)} | Searching {_whole_word_count} whole English words")

def _find_nearest(vector, top_n=10, exclude=None):
    """Find the top-N nearest whole words to an arbitrary vector."""
    exclude = {w.lower() for w in (exclude or [])}
    result_norm = vector / np.linalg.norm(vector)
    similarities = _normalized_embeddings @ result_norm
    ranked = np.argsort(similarities)[::-1]
    results = []
    for idx in ranked:
        if int(idx) in _skip_ids:
            continue
        token = bert_tokenizer.convert_ids_to_tokens(int(idx))
        if token in exclude:
            continue
        results.append((token, float(similarities[idx])))
        if len(results) >= top_n:
            break
    return results

def nearest_words(word, top_n=10):
    """Show the nearest neighbors to a word in BERT's embedding space."""
    print(f"\nNearest neighbors to '{word}' (from {_whole_word_count} whole-word tokens):")
    vec = get_word_embedding(word)
    results = _find_nearest(vec, top_n=top_n, exclude={word})
    for i, (token, sim) in enumerate(results, 1):
        print(f"  {i}. {token:15s} (similarity: {sim:.4f})")
    return results

def vector_arithmetic_demo(word1, operation, word2, operation2, word3, top_n=5):
    """
    Demonstrate vector arithmetic: word1 - word2 + word3
    Searches the entire BERT vocabulary (~30K tokens) for nearest neighbors.
    """
    print(f"\nComputing: {word1} {operation} {word2} {operation2} {word3}")
    
    v1 = get_word_embedding(word1)
    v2 = get_word_embedding(word2)
    v3 = get_word_embedding(word3)
    
    if operation == '-' and operation2 == '+':
        result_vector = v1 - v2 + v3
    else:
        print("Currently only supports 'word1 - word2 + word3' format")
        return
    
    results = _find_nearest(result_vector, top_n=top_n, exclude={word1, word2, word3})
    
    print(f"\nTop {top_n} most similar words (from {_whole_word_count} whole-word tokens):")
    for i, (word, sim) in enumerate(results, 1):
        print(f"  {i}. {word:15s} (similarity: {sim:.4f})")
    
    return results

def explore_axis(dim, top_n=10):
    """
    Show words at the extremes of a single embedding dimension.
    Helps reveal what a particular axis in the vector space might encode.
    """
    ndim = _embedding_matrix.shape[1]
    if not 0 <= dim < ndim:
        print(f"Dimension must be between 0 and {ndim - 1}")
        return
    
    axis_values = _embedding_matrix[:, dim]
    ranked = np.argsort(axis_values)
    
    # Collect top/bottom words, skipping filtered tokens
    high_words, low_words = [], []
    for idx in reversed(ranked):
        if int(idx) in _skip_ids:
            continue
        token = bert_tokenizer.convert_ids_to_tokens(int(idx))
        high_words.append((token, float(axis_values[idx])))
        if len(high_words) >= top_n:
            break
    for idx in ranked:
        if int(idx) in _skip_ids:
            continue
        token = bert_tokenizer.convert_ids_to_tokens(int(idx))
        low_words.append((token, float(axis_values[idx])))
        if len(low_words) >= top_n:
            break
    
    print(f"\n--- Dimension {dim} ---")
    print(f"\n  HIGH end (positive):")
    for i, (word, val) in enumerate(high_words, 1):
        print(f"    {i}. {word:15s} ({val:+.4f})")
    print(f"\n  LOW end (negative):")
    for i, (word, val) in enumerate(low_words, 1):
        print(f"    {i}. {word:15s} ({val:+.4f})")
    
    return high_words, low_words

def shared_direction(words, top_n=10):
    """
    Given a list of words, find the concept direction they share.
    Computes a centroid, subtracts the vocab-wide mean to isolate what
    makes this group *distinctive*, then finds nearest/farthest words.
    Also reports which embedding dimensions contribute most to the direction.
    """
    embeddings = np.array([get_word_embedding(w) for w in words])
    centroid = embeddings.mean(axis=0)
    
    # Subtract vocab mean to get the *distinctive* direction
    direction = centroid - _vocab_mean
    
    print(f"\nShared direction for: {words}")
    print(f"\n  Closest words (share this quality):")
    close = _find_nearest(direction, top_n=top_n, exclude=set(words))
    for i, (token, sim) in enumerate(close, 1):
        print(f"    {i}. {token:15s} (similarity: {sim:.4f})")
    
    print(f"\n  Farthest words (opposite end):")
    far = _find_nearest(-direction, top_n=top_n, exclude=set(words))
    for i, (token, sim) in enumerate(far, 1):
        print(f"    {i}. {token:15s} (similarity: {sim:.4f})")
    
    # Report the top dimensions that define this direction
    top_dims = np.argsort(np.abs(direction))[::-1][:5]
    print(f"\n  Top 5 dimensions defining this direction:")
    for dim in top_dims:
        sign = "+" if direction[dim] > 0 else "-"
        print(f"    dim {dim:3d}  ({sign}{abs(direction[dim]):.4f})")
    
    return direction, close, far

In [ ]:
# What words live near "python" in embedding space?
nearest_words('python')



Nearest neighbors to 'Python' (from 21745 whole-word tokens):
  1. crocodile       (similarity: 0.5949)
  2. snakes          (similarity: 0.5842)
  3. barbarian       (similarity: 0.5767)
  4. nottinghamshire (similarity: 0.5750)
  5. php             (similarity: 0.5680)
  6. dictator        (similarity: 0.5663)
  7. lizards         (similarity: 0.5663)
  8. peshawar        (similarity: 0.5632)
  9. fabio           (similarity: 0.5619)
  10. poisonous       (similarity: 0.5612)


[('crocodile', 0.5948618650436401),
 ('snakes', 0.584230363368988),
 ('barbarian', 0.5767242908477783),
 ('nottinghamshire', 0.5750338435173035),
 ('php', 0.5679692029953003),
 ('dictator', 0.5663412809371948),
 ('lizards', 0.5662696361541748),
 ('peshawar', 0.5631513595581055),
 ('fabio', 0.5618895888328552),
 ('poisonous', 0.561190128326416)]

In [ ]:
# What do reptiles have in common? Find the "reptile-ness" direction.
reptile_direction, close, far = shared_direction(
    ['snake', 'lizard', 'turtle', 'crocodile', 'alligator']
)

In [ ]:
# Now explore the strongest dimension from the "reptile-ness" direction.
# Does it look reptile-y, or is it encoding something more general?
top_reptile_dim = int(np.argsort(np.abs(reptile_direction))[::-1][0])
print(f"The strongest 'reptile-ness' dimension is {top_reptile_dim} — let's see what lives there:\n")
explore_axis(top_reptile_dim)

In [16]:
# Classic Example: Gender Analogy
vector_arithmetic_demo('king', '-', 'man', '+', 'woman')


Computing: king - man + woman

Top 5 most similar words:
  1. queen           (similarity: 0.7425)
  2. prince          (similarity: 0.7093)
  3. princess        (similarity: 0.7093)
  4. jordan          (similarity: 0.6402)
  5. jackson         (similarity: 0.6388)


[('queen', np.float32(0.74254704)),
 ('prince', np.float32(0.7093277)),
 ('princess', np.float32(0.70928264)),
 ('jordan', np.float32(0.640159)),
 ('jackson', np.float32(0.6388185)),
 ('brother', np.float32(0.6236417)),
 ('girl', np.float32(0.6199614)),
 ('mother', np.float32(0.60490626)),
 ('franco', np.float32(0.5950323)),
 ('father', np.float32(0.5877717)),
 ('sister', np.float32(0.58545065)),
 ('aunt', np.float32(0.56171286)),
 ('phelps', np.float32(0.5570026)),
 ('mussolini', np.float32(0.55530953)),
 ('uncle', np.float32(0.554166)),
 ('paris', np.float32(0.54546)),
 ('boy', np.float32(0.54496604)),
 ('napoleon', np.float32(0.53387517)),
 ('tyson', np.float32(0.51983666)),
 ('france', np.float32(0.5141543)),
 ('rome', np.float32(0.4923144)),
 ('london', np.float32(0.47165108)),
 ('berlin', np.float32(0.45070422)),
 ('italy', np.float32(0.4467885)),
 ('boxing', np.float32(0.41821003)),
 ('music', np.float32(0.4164229)),
 ('basketball', np.float32(0.3618493)),
 ('spain', np.float32(

In [12]:
# Historical Leaders Example
vector_arithmetic_demo('hitler', '-', 'germany', '+', 'italy')


Computing: hitler - germany + italy

Top 5 most similar words (from 24689 whole-word tokens):
  1. mussolini       (similarity: 0.6401)
  2. fascism         (similarity: 0.5700)
  3. italians        (similarity: 0.5602)
  4. michelangelo    (similarity: 0.5467)
  5. picasso         (similarity: 0.5450)


[('mussolini', 0.6401172876358032),
 ('fascism', 0.5700098872184753),
 ('italians', 0.560169517993927),
 ('michelangelo', 0.5466602444648743),
 ('picasso', 0.5449507236480713)]

In [ ]:
# National Capitals Example
vector_arithmetic_demo('paris', '-', 'france', '+', 'japan')


Computing: paris - france + japan

Top 5 most similar words (from 24689 whole-word tokens):
  1. japanese        (similarity: 0.5084)
  2. tokyo           (similarity: 0.4897)
  3. yokohama        (similarity: 0.4632)
  4. kyoto           (similarity: 0.4625)
  5. moscow          (similarity: 0.4575)


[('japanese', 0.5083581209182739),
 ('tokyo', 0.48966771364212036),
 ('yokohama', 0.4632266163825989),
 ('kyoto', 0.46254879236221313),
 ('moscow', 0.45750948786735535)]

In [14]:
# Try your own!
vector_arithmetic_demo('best', '-', 'good', '+', 'bad')


Computing: best - good + bad

Top 5 most similar words (from 24689 whole-word tokens):
  1. worst           (similarity: 0.5283)
  2. finest          (similarity: 0.3963)
  3. strongest       (similarity: 0.3809)
  4. hardest         (similarity: 0.3759)
  5. greatest        (similarity: 0.3746)


[('worst', 0.5282912254333496),
 ('finest', 0.3962985873222351),
 ('strongest', 0.3809419870376587),
 ('hardest', 0.375916451215744),
 ('greatest', 0.37458640336990356)]

---
## Part 3: Attention Mechanism - The Magic Happens Here

In [ ]:
def visualize_attention(text, layer=0, head=0):
    """
    Visualize attention weights for a given text.
    Shows which words the model pays attention to.
    """
    inputs = bert_tokenizer(text, return_tensors='pt')
    
    with torch.no_grad():
        outputs = bert_model(**inputs, output_attentions=True)
    
    attention = outputs.attentions[layer][0, head].numpy()
    tokens = bert_tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(attention, 
                xticklabels=tokens, 
                yticklabels=tokens,
                cmap='YlOrRd',
                cbar_kws={'label': 'Attention Weight'})
    plt.title(f'Attention Weights - Layer {layer}, Head {head}\n"{text}"')
    plt.xlabel('Key (attending to)')
    plt.ylabel('Query (attending from)')
    plt.tight_layout()
    plt.show()
    
    return attention, tokens

In [ ]:
visualize_attention("Michael Jordan plays basketball")

In [ ]:
visualize_attention("Michael Jackson made music")

In [ ]:
visualize_attention("The bank by the river")

In [ ]:
visualize_attention("Money in the bank")

In [ ]:
# Try your own!
visualize_attention('your sentence here')

---
## Part 4: Contextual Embeddings - Same Word, Different Meanings

In [ ]:
def compare_contextualized_embeddings(word, sentence1, sentence2):
    """
    Show how the same word gets different embeddings in different contexts.
    This is the KEY insight about attention!
    """
    print(f"\nAnalyzing the word '{word}' in different contexts:")
    print(f"  Context 1: '{sentence1}'")
    print(f"  Context 2: '{sentence2}'")
    
    def get_contextual_embedding(word, sentence):
        inputs = bert_tokenizer(sentence, return_tensors='pt')
        with torch.no_grad():
            outputs = bert_model(**inputs)
        
        tokens = bert_tokenizer.tokenize(sentence)
        word_tokens = bert_tokenizer.tokenize(word)
        
        for i, token in enumerate(tokens):
            if word.lower() in token.lower():
                embedding = outputs.last_hidden_state[0, i+1, :].numpy()
                return embedding
        
        return None
    
    emb1 = get_contextual_embedding(word, sentence1)
    emb2 = get_contextual_embedding(word, sentence2)
    
    if emb1 is not None and emb2 is not None:
        similarity = 1 - cosine(emb1, emb2)
        print(f"\nCosine similarity between the two embeddings: {similarity:.4f}")
        print(f"   (1.0 = identical, 0.0 = completely different)")
        
        if similarity > 0.9:
            print("   -> Very similar meanings in both contexts")
        elif similarity > 0.7:
            print("   -> Somewhat similar meanings")
        else:
            print("   -> Quite different meanings! Attention modified the representation.")
    
    return emb1, emb2, similarity

In [ ]:
# Polysemous Word 'Bank'
compare_contextualized_embeddings(
    'bank',
    'I deposited money at the bank',
    'We sat on the river bank'
)

In [ ]:
# Name 'Michael' with Different People
compare_contextualized_embeddings(
    'Michael',
    'Michael Jordan won six NBA championships',
    'Michael Jackson was the king of pop'
)

In [ ]:
# Word 'Apple'
compare_contextualized_embeddings(
    'apple',
    'I ate a delicious apple',
    'Apple released a new iPhone'
)

In [ ]:
# Try your own!
compare_contextualized_embeddings('word', 'sentence 1', 'sentence 2')

---
## Part 5: Next Token Prediction - It's More Than It Seems!

In [ ]:
# Load GPT-2 for generation
gpt2_tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
gpt2_model = GPT2LMHeadModel.from_pretrained('gpt2')
gpt2_model.eval()

print("GPT-2 model loaded!")

In [ ]:
def predict_next_tokens(prompt, top_k=10):
    """
    Show the top-k most likely next tokens.
    This demonstrates that 'predicting the next word' requires
    understanding context, syntax, semantics, and reasoning.
    """
    print(f"\nPredicting next token after: '{prompt}'")
    
    inputs = gpt2_tokenizer.encode(prompt, return_tensors='pt')
    
    with torch.no_grad():
        outputs = gpt2_model(inputs)
        predictions = outputs.logits
    
    next_token_logits = predictions[0, -1, :]
    next_token_probs = torch.softmax(next_token_logits, dim=0)
    
    top_probs, top_indices = torch.topk(next_token_probs, top_k)
    
    print(f"\nTop {top_k} most likely next tokens:")
    for i, (prob, idx) in enumerate(zip(top_probs, top_indices), 1):
        token = gpt2_tokenizer.decode([idx])
        print(f"  {i}. '{token}' (probability: {prob:.4f})")
    
    return top_probs, top_indices

In [ ]:
# Simple Completion
predict_next_tokens("The capital of France is")

In [ ]:
# Context-Dependent
predict_next_tokens("After winning the NBA championship, Michael Jordan")

In [ ]:
# Requires World Knowledge
predict_next_tokens("Python is a programming")

In [ ]:
# Try your own!
predict_next_tokens('your prompt here')

---
## Key Takeaways

1. **TOKENIZATION**: Text is broken into tokens (subwords) for processing

2. **EMBEDDINGS**: Words are represented as vectors in high-dimensional space
   - Similar concepts are closer together
   - Vector arithmetic captures semantic relationships

3. **ATTENTION**: The mechanism that allows context to modify meaning
   - Same word, different contexts -> different internal representations
   - The model learns which words to pay attention to

4. **LAYERS**: Deeper layers capture more abstract/complex patterns
   - Early layers: syntax and simple patterns
   - Later layers: reasoning, world knowledge, complex relationships

5. **"JUST PREDICTING THE NEXT WORD"**:
   Yes, but to do so accurately requires:
   - Understanding grammar and syntax
   - Modeling semantic relationships
   - Incorporating world knowledge
   - Reasoning about context
   
   Saying an LLM "just predicts the next word" is like saying a 
   chess grandmaster "just moves pieces" - technically true but 
   misses the sophisticated internal modeling that makes it possible!

---

**Thanks for exploring LLMs with me! Questions?**